In [6]:

!pip install -q langchain langchain-groq gspread pydantic

import os
from google.colab import userdata, auth

# Load Groq API key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize Groq LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

print("Groq LLM initialized successfully!")

# Google Sheets authentication
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create Google Sheet (run only once)
sh = gc.create("Sweet Crumbs Orders")

ws = sh.sheet1

# Add header row
ws.append_row([
    "name",
    "item",
    "qty",
    "date",
    "eggless",
    "total"
])

print("Google Sheet created successfully!")
print("Sheet URL:", sh.url)


Groq LLM initialized successfully!
Google Sheet created successfully!
Sheet URL: https://docs.google.com/spreadsheets/d/1ZXb8HfrMD7TqU2g5Jp7tFW0IgnwrLvZoJ5bxtR_4dnw


In [7]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

# Bakery menu
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

# Bakery opening hours
HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}

# Pydantic Order Model
class Order(BaseModel):
    """A cake order taken from a customer's chat message."""

    name: str = Field(description="Customer's first name")

    item: Literal[
        "plum cake",
        "chocolate cake",
        "cupcake"
    ] = Field(description="Menu item")

    qty: int = Field(description="Quantity", ge=1)

    date: str = Field(
        description="Pickup date as the customer said it"
    )

    eggless: bool = Field(
        description="True if customer requested eggless"
    )


# Tool 1: Get Price
@tool
def get_price(item: str, qty: int = 1) -> str:
    """
    Get the total price in rupees for a menu item.

    Args:
        item: plum cake, chocolate cake or cupcake
        qty: Number of items
    """

    price = MENU.get(item.lower().strip())

    if price is None:
        return (
            f"'{item}' is not on the menu. "
            f"We have: {', '.join(MENU)}."
        )

    if qty < 1:
        return "Quantity must be at least 1."

    return f"{qty} x {item} = Rs {price * qty}"


# Tool 2: Check Opening Hours
@tool
def is_open(day: str) -> str:
    """Check the bakery opening hours for a day of the week."""

    hours = HOURS.get(day.lower().strip())

    if hours:
        return f"Open {hours} on {day}."

    return f"'{day}' is not a day of the week."


# Tool 3: Save Order
@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """
    Save a confirmed cake order to the Google Orders sheet.

    Use only when customer name, item, quantity and pickup
    date are available.
    """

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

    except Exception as e:
        return f"Could not save order: {e}"

    total = MENU[order.item] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.eggless else "no",
        total
    ])

    return (
        f"Saved: {order.qty} x {order.item} "
        f"for {order.name} on {order.date}, "
        f"Rs {total}."
    )

print("Order model and all three tools created successfully!")

Order model and all three tools created successfully!


In [8]:
from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are a friendly order assistant for Sweet Crumbs Home Bakery.

FACTS:
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
1. Always use get_price for prices.
2. Always use is_open for opening hours.
3. Never guess prices or timings.
4. Before calling log_order, collect:
   customer name, item, quantity and pickup date.
5. If any information is missing, ask the customer.
6. Never save incomplete orders.
7. After saving, confirm the order and total.

FORMAT:
Keep answers under 60 words.
Reply in friendly English or Telugu,
matching the customer's language.
"""

# Create agent
agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

# Chat history
history = []

def chat(text: str) -> str:
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)

        return "Sorry, I'm a little busy. Please try again in a minute."


# Test the agent
print("Customer:", "Are you open on Sunday?")
print("Assistant:", chat("Are you open on Sunday?"))

print("\nCustomer:", "How much are 2 chocolate cakes?")
print("Assistant:", chat("How much are 2 chocolate cakes?"))

print("\nCustomer:", "I'm Ravi. Book 2 eggless chocolate cakes for 24 Dec.")
print("Assistant:", chat(
    "I'm Ravi. Book 2 eggless chocolate cakes for 24 Dec."
))

Customer: Are you open on Sunday?
Assistant: Yes! We’re open on Sunday from 10 am to 6 pm. Let me know if you’d like to place an order.

Customer: How much are 2 chocolate cakes?
Assistant: Two chocolate cakes cost ₹1,300 in total. If you’d like to order, just let me know your name, the pickup date, and we’ll get everything set up!

Customer: I'm Ravi. Book 2 eggless chocolate cakes for 24 Dec.
Assistant: All set, Ravi! Your 2 eggless chocolate cakes are booked for pickup on 24 Dec. Total: ₹1,300. We’ll see you then—thanks for choosing Sweet Crumbs!


In [9]:
print("========== AGENT EXECUTION TRACE ==========")

for m in history:
    m.pretty_print()

print("\n========== FINAL ANSWER ==========")

print(history[-1].content)

========== AGENT EXECUTION TRACE ==========
================================ Human Message =================================

Are you open on Sunday?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_cc6d9c0f-3c71-4868-9172-ad5739513d25)
 Call ID: fc_cc6d9c0f-3c71-4868-9172-ad5739513d25
  Args:
    day: Sunday
================================= Tool Message =================================
Name: is_open

Open 10 am to 6 pm on Sunday.
================================== Ai Message ==================================

Yes! We’re open on Sunday from 10 am to 6 pm. Let me know if you’d like to place an order.
================================ Human Message =================================

How much are 2 chocolate cakes?
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_ee122afa-273e-4010-a527-4501cb338453)
 Call ID: fc_ee122afa-273e-4010-a527-4501cb338453
  Args:
    item: cho

In [10]:
test_questions = [
    "Do you have donuts?",
    "I want 2 plum cakes.",
    "I want 0 cupcakes.",
    "What are your Monday opening hours?",
    "How much are 6 cupcakes?"
]

for question in test_questions:
    print("\nCustomer:", question)
    print("Assistant:", chat(question))


Customer: Do you have donuts?
Assistant: Sorry, we don’t offer donuts. We have a variety of cakes and cupcakes—eggless options available. Let me know if you’d like to try something else!

Customer: I want 2 plum cakes.
Assistant: Sure! Could you share your name and the pickup date you’d like? I’ll also fetch the price for 2 plum cakes.

Customer: I want 0 cupcakes.
Assistant: Got it—just to confirm: what’s your name and the pickup date you’d like? Also, did you mean 0 cupcakes or perhaps a different quantity? Let me know so I can set it up for you!

Customer: What are your Monday opening hours?
Assistant: We’re open on Monday from 9 am to 8 pm. Let us know if you’d like to place an order!

Customer: How much are 6 cupcakes?
Assistant: Six cupcakes cost ₹480 in total. Let me know if you’d like to place an order!


In [11]:
# Display all saved orders
import pandas as pd

data = ws.get_all_records()

df = pd.DataFrame(data)

display(df)

,name,item,qty,date,eggless,total
0,Ravi,chocolate cake,2,24 Dec,yes,1300
